#### Import DuckDB

In [34]:
import duckdb

#### Create the database

In [35]:
conn = duckdb.connect("insurance.duckdb")

#### Load CSV files into the database

In [36]:
conn.execute(""" 
CREATE OR REPLACE TABLE dim_adjusters AS 
SELECT *
FROM read_csv_auto('data/dim_adjusters.csv');
""")

conn.execute(""" 
CREATE OR REPLACE TABLE dim_customers AS 
SELECT *
FROM read_csv_auto('data/dim_customers.csv');
""")

conn.execute(""" 
CREATE OR REPLACE TABLE dim_policies AS 
SELECT *
FROM read_csv_auto('data/dim_policies.csv');
""")

conn.execute(""" 
CREATE OR REPLACE TABLE fact_claims AS 
SELECT *
FROM read_csv_auto('data/fact_claims.csv');
""")

#### Test and verify that the data was loaded in the database

In [37]:
from IPython.display import display, Markdown

display(Markdown("Check that the tables are in the DB"))
display( 
    conn.sql(""" 
    SHOW TABLES;
    """).df() 
)

display(Markdown("Quick data snapshot test for **dim_adjusters** table: "))
display( 
    conn.sql(""" 
    SELECT * 
    FROM dim_adjusters 
    LIMIT 5;
    """).df()
)

display(Markdown("Quick data snapshot test for **dim_customers** table: "))
display( 
    conn.sql(""" 
    SELECT * 
    FROM dim_customers
    LIMIT 5;
    """).df()
)

display(Markdown("Quick data snapshot test for **dim_policies** table: "))
display( 
    conn.sql(""" 
    SELECT * 
    FROM dim_policies
    LIMIT 5;
    """).df()
)

display(Markdown("Quick data snapshot test for **fact_claims** table: "))
display( 
    conn.sql(""" 
    SELECT * 
    FROM fact_claims
    LIMIT 5;
    """).df()
)

Check that the tables are in the DB

,name
0,dim_adjusters
1,dim_customers
2,dim_policies
3,fact_claims


Quick data snapshot test for **dim_adjusters** table: 

,adjuster_id,adjuster_name,region,hire_date
0,ADJ-001,J. Romero,Northeast,2017-03-06
1,ADJ-002,A. Singh,West,2015-06-09
2,ADJ-003,M. Chen,Midwest,2019-02-01
3,ADJ-004,L. Garcia,Midwest,2018-02-13
4,ADJ-005,K. Patel,Northeast,2012-08-08


Quick data snapshot test for **dim_customers** table: 

,customer_id,first_name,last_name,date_of_birth,email,signup_date
0,CUST-20001,Jose,Moore,1991-10-11,jose.moore1@mail.com,2017-02-16
1,CUST-20002,Sandra,Flores,1992-09-06,sandra.flores2@mail.com,2016-01-21
2,CUST-20003,Jose,Rodriguez,1962-09-21,jose.rodriguez3@mail.com,2022-10-11
3,CUST-20004,Maria,Reyes,1985-07-17,maria.reyes4@mail.com,2021-11-26
4,CUST-20005,Sandra,Reyes,1951-05-07,sandra.reyes5@mail.com,2015-05-27


Quick data snapshot test for **dim_policies** table: 

,policy_id,customer_id,policy_type,state,coverage_limit,premium_amount,effective_date,expiration_date
0,POL-100001,CUST-20171,AUTO,IL,30322.92,320.95,2022-04-22,2023-04-22
1,POL-100002,CUST-20026,Renters,GA,40109.20,1273.39,2023-04-30,2024-04-29
2,POL-100003,CUST-20223,Home,wa,373132.34,9165.18,2022-12-02,2023-12-02
3,POL-100004,CUST-20441,Renters,GA,10830.75,266.67,2023-10-17,2024-10-16
4,POL-100005,CUST-20470,Health,TX,47381.30,1683.17,2022-03-22,2023-03-22


Quick data snapshot test for **fact_claims** table: 

,claim_id,policy_id,adjuster_id,claim_date,cause_of_loss,claim_amount,claim_status
0,CLM-10692,POL-100677,ADJ-007,2023-10-29,Collision,14985.34,In Review
1,CLM-10947,POL-100044,ADJ-010,2023-12-22,Emergency Visit,16307.17,Closed
2,CLM-11008,POL-100392,ADJ-008,2023-05-01,Outpatient Procedure,10643.69,Denied
3,CLM-10810,POL-100376,ADJ-002,2023-09-16,Fire,23605.88,Denied
4,CLM-11370,POL-100715,ADJ-008,2023-03-27,Vandalism,13189.58,Closed


## SQL **|** TECHNICAL INTERVIEW PREP

### Section **A** - Joins 

#### A - Q1

**Write a query that returns each claim with the customer's full name (`first_name + last_name`), the policy type, and the claim amount. Only include claims where `claim_amount` is greater than 5,000.**

> 💡 **Hint:** You'll need to join `fact_claims → dim_policies → dim_customers`. Think about which join type to use and why.


In [38]:
conn.sql(""" 
    SELECT cl.claim_id,
           CONCAT(cu.first_name || ' ' || cu.last_name) AS full_name,
           po.policy_type,
           cl.claim_amount
             
    FROM fact_claims cl
         
    JOIN dim_policies po ON cl.policy_id = po.policy_id
    JOIN dim_customers cu ON po.customer_id = cu.customer_id 
    WHERE cl.claim_amount > 5000 
    ORDER BY cl.claim_amount DESC;
    """).df()

,claim_id,full_name,policy_type,claim_amount
0,CLM-10780,Andres Davis,Life,1354169.90
1,CLM-10664,Jose Martinez,Home,1060810.63
2,CLM-10314,Carlos Smith,Life,1053786.91
3,CLM-10292,Linda Miller,life,931815.80
4,CLM-11147,Jose Sanchez,Life,876942.69
...,...,...,...,...
1178,CLM-10886,Michael Perez,Health,5046.81
1179,CLM-10993,Robert Brown,Renters,5027.52
1180,CLM-10580,Linda Rodriguez,Renters,5018.51
1181,CLM-11396,Robert Martinez,Auto,5010.35


**A - Q1  | Explanation**
> <br>The solution requires to display all claims so I chose my fact claims table as my driving table, the rest of the information comes from another tables. I decided to use an INNER JOIN because the query is interested in claims that have matching policy and customer information. Finally I filter the rows where the claim amount is greater than 5000.
<br>
<br>

#### A - Q2

**Find all policies that have NO claims at all. Return `policy_id`, `policy_type`, and `customer_id`.**

> 💡 **Hint:** Think about which type of JOIN is designed specifically to find rows in one table with no match in another.


In [39]:
conn.sql(""" 
    SELECT po.policy_id,
           po.policy_type,
           po.customer_id
             
    FROM dim_policies po
         
    LEFT JOIN fact_claims cl ON po.policy_id = cl.policy_id
    WHERE cl.claim_id IS NULL;
         
    """).df()

,policy_id,policy_type,customer_id
0,POL-100083,Renters,CUST-20347
1,POL-100200,Life,CUST-20462
2,POL-100225,Home,CUST-20163
3,POL-100228,Renters,CUST-20259
4,POL-100281,Health,CUST-20403
...,...,...,...
108,POL-100400,Renters,CUST-20372
109,POL-100409,Renters,CUST-20437
110,POL-100520,Auto,CUST-20254
111,POL-100617,Home,CUST-20182


**A - Q2  | Explanation**
> <br>For this solution I selected dim_policies as my driving table, then I chose LEFT JOIN in order to keep all my policies and join the claims table, then I just filtered in order to retrieve all the policies with no claims (rows with NULL claim_id values).
<br>
<br>

#### A - Q3

**Write a query showing each `adjuster_name`, their `region`, and the total number of claims they've handled. Include adjusters who have handled zero claims. Order by total claims descending.**

> 💡 **Hint:** You need dim_adjusters and fact_claims. Consider which JOIN keeps adjusters with zero claims in the result.


In [40]:
conn.sql(""" 
    SELECT ad.adjuster_name,
           ad.region,
           COUNT(cl.claim_id) AS total_claims
             
    FROM dim_adjusters ad
         
    LEFT JOIN fact_claims cl ON ad.adjuster_id = cl.adjuster_id
    
    GROUP BY  ad.adjuster_name,
              ad.region
         
    ORDER BY total_claims DESC
   ;
         
    """).df()

,adjuster_name,region,total_claims
0,C. Dubois,South,168
1,M. Chen,Midwest,168
2,J. Romero,Northeast,152
3,T. Nguyen,West,152
4,A. Singh,West,151
5,B. Okafor,South,149
6,S. Lee,South,144
7,K. Patel,Northeast,139
8,R. Johnson,South,133
9,L. Garcia,Midwest,131


**A - Q3  | Explanation**
> <br>I chose LEFT JOIN again in order to retrieve all the adjusters, then I count the claim IDs because if I count(*) the null claims will be ignored. So I counted all claims and group them by adjuster and their region.
<br>
<br>

#### A - Q4

**This is a referential integrity check: find all claims in `fact_claims` that reference a `policy_id` that does NOT exist in `dim_policies`. Return `claim_id` and the orphaned `policy_id`.**

> 💡 **Hint:** This is a real QA query — think about what JOIN pattern surfaces rows with no match on the right side.

In [41]:
conn.sql(""" 
    SELECT cl.claim_id,
           cl.policy_id as orphaned_policy_id
    FROM fact_claims cl
    LEFT JOIN dim_policies po ON cl.policy_id = po.policy_id
    WHERE po.policy_id IS NULL;
         
    """).df()

,claim_id,orphaned_policy_id
0,CLM-10025,POL-981666
1,CLM-11356,POL-933430
2,CLM-10551,POL-956908
3,CLM-10444,POL-960927
4,CLM-10561,POL-966584
5,CLM-10675,POL-910584
6,CLM-10447,POL-985206
7,CLM-10784,POL-928415
8,CLM-10145,POL-947932
9,CLM-11459,POL-952504


**A - Q4  | Explanation**
> <br>This follows the same pattern I reasoned in Q2, since the problem indicates that we need to include all claims I used fact_claims as my driving table, then I also selected the policies within claims table, the second part was to LEFT JOIN dim_policies and filter the ones that are NULL or are not included in claims table. 
<br>
<br>

### Section **B** - Window Functions

#### B - Q1

**For each `policy_type`, rank claims by `claim_amount` from highest to lowest. Return `claim_id`, `policy_type`, `claim_amount`, and the rank. Only return the top 3 claims per policy type.**

> 💡 **Hint:** ROW_NUMBER() or RANK() OVER (PARTITION BY ... ORDER BY ...) — think about what PARTITION BY means here.

In [42]:
conn.sql(""" 
    
    SELECT claim_id,
           policy_type,
           claim_amount,
           claim_rank
         
    FROM (
              SELECT cl.claim_id,
                     po.policy_type,
                     cl.claim_amount,
                     RANK() OVER(PARTITION BY po.policy_type ORDER BY cl.claim_amount DESC) AS claim_rank
              FROM fact_claims cl
              JOIN dim_policies po ON cl.policy_id = po.policy_id) 
    WHERE claim_rank <= 3;
         
    """).df()

,claim_id,policy_type,claim_amount,claim_rank
0,CLM-10796,AUTO,149585.96,1
1,CLM-11132,AUTO,110235.94,2
2,CLM-10052,AUTO,36049.03,3
3,CLM-10664,Home,1060810.63,1
4,CLM-11040,Home,609777.15,2
5,CLM-11086,Home,498344.60,3
6,CLM-11166,home,444442.90,1
7,CLM-10671,home,392819.31,2
8,CLM-10255,home,387605.35,3
9,CLM-10167,Renters,110111.49,1


**B - Q1  | Explanation**
> <br>Chose the fact_claims as my driving table and joined the matching values from the dim_policies table, in order to rank the claim_amount for each policy_type I add a new column using the RANK window function and then I used a sub-query in order to filter the RANK to just show me the 3 highests amounts of each policy type.
<br>
<br>

#### B - Q2

**For each claim, show the `claim_id`, `claim_date`, `claim_amount`, and the `claim_amount` of the PREVIOUS claim (ordered by `claim_date`). Call that column prev_claim_amount.**

> 💡 **Hint:** LAG() lets you look at the previous row's value within a partition or ordered set.

In [43]:
conn.sql(""" 
    
    SELECT claim_id,
           claim_date,
           claim_amount,
           LAG(claim_amount) OVER(ORDER BY claim_date) AS prev_claim_amount
    FROM fact_claims;
         
    """).df()

,claim_id,claim_date,claim_amount,prev_claim_amount
0,CLM-10108,2022-01-14,337427.36,NaN
1,CLM-10674,2022-01-18,8272.36,337427.36
2,CLM-10084,2022-01-28,1703.99,8272.36
3,CLM-11444,2022-02-07,10392.21,1703.99
4,CLM-10546,2022-02-12,100995.24,10392.21
...,...,...,...,...
1519,CLM-11083,NaT,5111.99,50001.68
1520,CLM-10707,NaT,21148.39,5111.99
1521,CLM-10350,NaT,1666.27,21148.39
1522,CLM-11357,NaT,353859.89,1666.27


**B - Q2  | Explanation**
> <br> As per the problem instructions I selected the claims information (claim_id, claim_date, claim_amount). The second part was just add another column using the LAG window function in order to show the previous row of the claim_amount column and compare both. 
<br>
<br>

#### B - Q3

**Write a query that shows each claim ordered by `claim_date`, with a running total of `claim_amount` as a new column called running_total. Only include claims where `claim_amount` > 0.**

> 💡 **Hint:** SUM() OVER (ORDER BY ...) gives you a cumulative/running sum — no GROUP BY needed.

In [44]:
conn.sql(""" 
    
    SELECT claim_id,
           claim_date,
           claim_amount,
           SUM(claim_amount) OVER(ORDER BY claim_date) AS running_total
           
    FROM fact_claims
    WHERE claim_amount > 0
    ORDER BY claim_date;
         
    """).df()

,claim_id,claim_date,claim_amount,running_total
0,CLM-10108,2022-01-14,337427.36,3.374274e+05
1,CLM-10674,2022-01-18,8272.36,3.456997e+05
2,CLM-10084,2022-01-28,1703.99,3.474037e+05
3,CLM-11444,2022-02-07,10392.21,3.577959e+05
4,CLM-10546,2022-02-12,100995.24,4.587912e+05
...,...,...,...,...
1506,CLM-11083,NaT,5111.99,1.099493e+08
1507,CLM-10707,NaT,21148.39,1.099493e+08
1508,CLM-10350,NaT,1666.27,1.099493e+08
1509,CLM-10032,NaT,24472.68,1.099493e+08


**B - Q3  | Explanation**
> <br> I selected each claim with its corresponding date and amount then I added an extra column where the cummulative total of the claim amount is shown ordered by claim date. 
<br>
<br>

#### B - Q4

**Show the top 2 most expensive claims per state (use `dim_policies` for state). If two claims have the same amount, they should share the same rank. What window function handles ties correctly here, and why is ROW_NUMBER() the wrong choice?**

> 💡 **Hint:** Think about what happens when two rows are equal — ROW_NUMBER() assigns them different numbers arbitrarily. Is that correct behavior here?

In [45]:
conn.sql(""" 
    
    SELECT * 
         FROM(
              SELECT cl.claim_id,
                     cl.claim_amount,
                     po.state,
                     RANK () OVER(PARTITION BY po.state ORDER BY cl.claim_amount DESC) AS rnk
              FROM fact_claims cl
              JOIN dim_policies po ON cl.policy_id = po.policy_id)
         WHERE rnk <= 2
         
    """).df()

,claim_id,claim_amount,state,rnk
0,CLM-10780,1354169.90,AZ,1
1,CLM-10894,518775.33,AZ,2
2,CLM-11020,378957.16,fl,1
3,CLM-10327,312055.65,fl,2
4,CLM-10664,1060810.63,CA,1
5,CLM-11311,483819.65,CA,2
6,CLM-10650,479472.62,TX,1
7,CLM-10568,375249.73,TX,2
8,CLM-10906,72631.92,tx,1
9,CLM-10615,37311.83,tx,2


**B - Q4  | Explanation**
> <br> In this case I used the RANK window function because it ranks with the same value if a row has the same value or amount. ROW_NUMBER would assign a sequential value to each row even if the rows have the same value. 
<br>
<br>

### Section **C** - Validation & Reconciliation Queries

#### C - Q1

**Write a single query that counts null values in `claim_date`, `claim_amount` and `adjuster_id` across `fact_claims` — all in one result row with three columns: null_claim_date, null_claim_amount, null_adjuster_id.**


> 💡 **Hint:** Think about what happens when two rows are equal — ROW_NUMBER() assigns them different numbers arbitrarily. Is that correct behavior here?

In [46]:
conn.sql(""" 
    
    SELECT 
         SUM(CASE WHEN claim_date IS NULL THEN 1 ELSE 0 END) AS null_claim_date,
         SUM(CASE WHEN claim_amount IS NULL THEN 1 ELSE 0 END) AS null_claim_amount,
         SUM(CASE WHEN adjuster_id IS NULL THEN 1 ELSE 0 END) AS null_adjuster_id

    FROM fact_claims;
         
    """).df()

,null_claim_date,null_claim_amount,null_adjuster_id
0,59.0,0.0,0.0


**C - Q1  | Explanation**
> <br> In order to summarize or count found NULL values I created CASE WHEN clauses for each column. 
<br>
<br>

#### C - Q2

**Find all `claim_id` values in `fact_claims` that appear more than once. Return `claim_id` and how many times it appears. Order by occurrences descending.** 


> 💡 **Hint:** GROUP BY the business key, then filter with HAVING.

In [47]:
conn.sql(""" 
    
    SELECT 
         claim_id, 
         COUNT(claim_id) AS claims_count
    FROM fact_claims
    GROUP BY claim_ID
    HAVING COUNT(claim_id) > 1
    ORDER BY claims_count DESC;
         
    """).df()

,claim_id,claims_count
0,CLM-10843,2
1,CLM-10887,2
2,CLM-10335,2
3,CLM-10588,2
4,CLM-10138,2
5,CLM-11176,2
6,CLM-11301,2
7,CLM-10306,2
8,CLM-10652,2
9,CLM-11021,2


**C - Q2  | Explanation**
> <br> First I selected claims_id, this is the column where we want to verify if there are NULL values, after grouping I applied the HAVING clause in order to filter claim_ids that appeared more than 1 time.
<br>
<br>

#### C - Q3

**Find all claims where the `claim_amount` exceeds the `coverage_limit` on the linked policy. Return `claim_id`, `policy_id`, `claim_amount`, `coverage_limit`, and the overage amount (the difference). Order by overage descending.** 


> 💡 **Hint:** This requires a JOIN between `fact_claims` and `dim_policies` — a cross-table business rule check, which is the most realistic type of validation query.

In [48]:
conn.sql(""" 
    
    SELECT cl.claim_id,
           cl.policy_id,
           cl.claim_amount,
           po.coverage_limit,
           cl.claim_amount - po.coverage_limit AS overage_amount
             
           
    FROM fact_claims cl
    JOIN dim_policies po ON cl.policy_id = po.policy_id
    WHERE cl.claim_amount > po.coverage_limit
    ORDER BY overage_amount DESC;
         
    """).df()

,claim_id,policy_id,claim_amount,coverage_limit,overage_amount
0,CLM-10780,POL-100788,1354169.90,755049.50,599120.40
1,CLM-10664,POL-100610,1060810.63,660661.24,400149.39
2,CLM-10314,POL-100451,1053786.91,667502.11,386284.80
3,CLM-11147,POL-100582,876942.69,579317.35,297625.34
4,CLM-11085,POL-100114,687735.99,405618.79,282117.20
5,CLM-11199,POL-100448,793326.16,553920.11,239406.05
6,CLM-10292,POL-100612,931815.80,734788.69,197027.11
7,CLM-11086,POL-100717,498344.60,350677.41,147667.19
8,CLM-11040,POL-100088,609777.15,543013.38,66763.77
9,CLM-10132,POL-100486,145486.87,82202.57,63284.30


**C - Q3  | Explanation**
> <br> I joined fact_claims with dim_policies in order to compare each claim_amount with the corresponding policys coverage_limit. I filtered just the values where claim_amount is greater than coverage_limit and then add another column (overage_amount) to display the difference.
<br>
<br>

#### C - Q4

**Find all claims where the `claim_date` falls OUTSIDE the policy's active period (before `effective_date` OR `after expiration_date`). Return `claim_id`, `claim_date`, `effective_date`, and `expiration_date`.** 


> 💡 **Hint:** Join fact_claims to dim_policies and compare dates. Exclude rows where claim_date is NULL

In [49]:
conn.sql(""" 
    
    SELECT cl.claim_id,
           cl.claim_date,
           po.effective_date, 
           po.expiration_date
    FROM fact_claims cl
    JOIN dim_policies po ON cl.policy_id = po.policy_id
    WHERE (cl.claim_date < po.effective_date
    OR cl.claim_date > po.expiration_date)
    AND claim_date IS NOT NULL;
         
    """).df()

,claim_id,claim_date,effective_date,expiration_date
0,CLM-10483,2024-11-21,2023-11-10,2024-11-09
1,CLM-11198,2023-08-28,2022-07-21,2023-07-21
2,CLM-10718,2023-11-18,2022-06-21,2023-06-21
3,CLM-11272,2024-10-15,2023-05-01,2024-04-30
4,CLM-10947,2023-12-22,2022-10-23,2023-10-23
...,...,...,...,...
111,CLM-11094,2025-06-02,2023-11-27,2024-11-26
112,CLM-10502,2023-08-24,2022-07-06,2023-07-06
113,CLM-11029,2023-08-13,2022-08-04,2023-08-04
114,CLM-10292,2023-07-12,2022-04-10,2023-04-10


**C - Q4  | Explanation**
> <br> I filtered claims occurring before the policy starts or after it expires and also included a separated condition to exclude NULL claim_date rows because we should not evaluate missing dates.
<br>
<br>

#### C - Q5

**Write a query showing the loss ratio per `policy_type` (total claim_amount / total premium_amount). Only include `claim_amount` > 0. Round loss ratio to 3 decimal places. Order by loss_ratio descending. What does a loss ratio above 1.0 mean for an insurance company?** 


> 💡 **Hint:** You need to aggregate across both `fact_claims` and `dim_policies`. Also think about what the business answer means — interviewers often ask 'so what does this number tell you?'

In [50]:
conn.sql(""" 
     
    WITH formatted_table AS (
         
         SELECT UPPER(TRIM(po.policy_type)) AS policy_type,
                cl.claim_amount AS claim_amount,
                po.premium_amount AS premium_amount
         FROM fact_claims cl
         JOIN dim_policies po ON cl.policy_id = po.policy_id 

         ),
         
         sum_table AS (
         SELECT policy_type,
                SUM(claim_amount) AS total_claim_amount,
                SUM(premium_amount) AS total_premium_amount
         FROM formatted_table
         WHERE claim_amount > 0
         GROUP BY policy_type)
         
         SELECT policy_type,
                ROUND(total_claim_amount / total_premium_amount,3) AS loss_ratio
         FROM sum_table
         ORDER BY loss_ratio DESC;
         
    """).df()

,policy_type,loss_ratio
0,HOME,14.237
1,AUTO,13.732
2,LIFE,13.390
3,HEALTH,13.138
4,RENTERS,13.076


**C - Q5  | Explanation**
> <br> For this problem I used CTEs for better organization and readability. If the loss ratio is greater than 1 it means that the insurance company is paying more money from claims than it is collecting from premium amount. That is not good and need to be investigated.
<br>
<br>

#### C - Q6

**`dim_policies.policy_type` has inconsistent casing (e.g. 'life', 'LIFE', 'Life'). Write a query that counts policies per `policy_type`, standardized to uppercase. Show what the result looks like WITHOUT standardization first (to demonstrate the bug), then with it fixed.** 


> 💡 **Hint:** UPPER(TRIM(`policy_type`)) before GROUP BY — but write the broken version first to show you understand why the fix is needed.

Query WITHOUT standardization / Inconsistent Data / Bug Demonstration

In [51]:
conn.sql(""" 
     SELECT policy_type AS inconsistent_policy_type,
            COUNT(policy_id) as policy_cnt
     FROM dim_policies 
     GROUP BY inconsistent_policy_type 
     ORDER BY policy_cnt DESC;
        
         
    """).df()

,inconsistent_policy_type,policy_cnt
0,Renters,153
1,Health,147
2,Home,142
3,Life,133
4,Auto,132
5,life,13
6,home,12
7,auto,12
8,renters,11
9,AUTO,11


Query WITH standardization / Standardized Data / Fixed Policy Type Names

In [52]:
conn.sql(""" 
       SELECT UPPER(TRIM(policy_type)) AS standardized_policy_type,
              COUNT(policy_id) as policy_cnt
       FROM dim_policies 
       GROUP BY standardized_policy_type
       ORDER BY policy_cnt DESC;
 """).df()

,standardized_policy_type,policy_cnt
0,RENTERS,167
1,HEALTH,162
2,HOME,161
3,AUTO,155
4,LIFE,155


**C - Q6  | Explanation**
> <br> I divided my solution in 2 parts. The first query groups the amount of policies accross all the inconsistent policy types. THe second query is standardizing policy types to be upper case and without spaces and then grouped the count.
<br>
<br>